# 04 — 配置档案（Profiles）

**来源:** [LangChain Docs — Deep Agents / Profiles](https://docs.langchain.com/oss/python/deepagents/profiles)

配置档案（Profiles）让你将特定 Provider 或特定模型的默认配置打包，当选择对应模型时 Deep Agents 自动应用。

本笔记涵盖：
- **HarnessProfile** — 影响 Prompt、工具、中间件的 Harness 级配置
- **ProviderProfile** — 影响模型构造参数（`init_chat_model` kwargs）
- 注册键的查找顺序与合并语义
- 从 YAML/JSON 加载配置
- 作为 Plugin 分发 Profile

## 1. HarnessProfile（Harness 级配置）

`HarnessProfile` 描述 Prompt 组装、工具可见性、中间件和默认子 Agent 的调整，在 Chat Model 构造后由 `create_deep_agent` 应用。

In [ ]:
from dotenv import load_dotenv
load_dotenv()
from deepagents import (
    GeneralPurposeSubagentProfile,
    HarnessProfile,
    register_harness_profile,
)

# 为 gpt-5.4 注册一个 HarnessProfile
register_harness_profile(
    "openai:gpt-5.4",
    HarnessProfile(
        system_prompt_suffix="Respond in under 100 words.",
        excluded_tools={"execute"},
        excluded_middleware={"SummarizationMiddleware"},
        general_purpose_subagent=GeneralPurposeSubagentProfile(enabled=False),
    ),
)

print("HarnessProfile 已注册到 openai:gpt-5.4。")
print("效果：减少输出长度、禁用 execute 工具、移除摘要中间件、禁用通用子 Agent。")


HarnessProfile 已注册到 openai:gpt-5.4。
效果：减少输出长度、禁用 execute 工具、移除摘要中间件、禁用通用子 Agent。


### 1.1 HarnessProfile 字段说明

| 字段 | 类型 | 说明 |
|------|------|------|
| `base_system_prompt` | string | 替换 Deep Agents 的基础系统 prompt（对应 Prompt assembly 中的 CUSTOM） |
| `system_prompt_suffix` | string | 追加到组装后的 base prompt 末尾（对应 SUFFIX），应用于主 Agent 和所有子 Agent |
| `tool_description_overrides` | Mapping[str, str] | 按工具名覆盖工具描述 |
| `excluded_tools` | frozenset[str] | 从工具集中移除指定工具（匹配工具名） |
| `excluded_middleware` | frozenset[type\|str] | 从中间件栈中移除指定中间件（类或字符串名） |
| `extra_middleware` | Sequence[AgentMiddleware] | 追加额外中间件到每一层栈 |
| `general_purpose_subagent` | GeneralPurposeSubagentProfile | 禁用、重命名或修改通用子 Agent 的 Prompt |

### 1.2 Prompt 覆盖规则

- 用户传入的 `system_prompt=` 始终在最前面
- `system_prompt_suffix` 始终在末尾
- 子 Agent 会对其自己的模型重新执行 Profile 解析
- `base_system_prompt` 仅替换内部默认 prompt，不影响用户传入的 `system_prompt=`

### 1.3 Excluded Middleware 注意事项

列出 `FilesystemMiddleware`、`SubAgentMiddleware` 或内部权限中间件会抛出 `ValueError`（它们是必需的脚手架）。

要隐藏工具而不移除中间件，使用 `excluded_tools` 替代。

`excluded_middleware` 接受两种形式：
- 中间件类（按精确类型匹配）
- 字符串名（匹配 `AgentMiddleware.name`），如 `"SummarizationMiddleware"`
- 模块路径引用，如 `"my_pkg.middleware:TelemetryMiddleware"`

## 2. ProviderProfile（Provider 级配置）

`ProviderProfile` 声明如何构造 Chat Model，仅在使用 `provider:model` 字符串创建 Agent 时生效（传入已预配置的模型实例时不生效）。

In [ ]:
from deepagents import ProviderProfile, register_provider_profile

# 为所有 OpenAI 模型设置 temperature=0
register_provider_profile(
    "openai",
    ProviderProfile(init_kwargs={"temperature": 0}),
)

# 为特定模型额外设置 reasoning_effort
register_provider_profile(
    "openai:gpt-5.4",
    ProviderProfile(init_kwargs={"reasoning_effort": "medium"}),
)

print("ProviderProfile 已注册。")
print("gpt-5.4 将使用: temperature=0 (继承) + reasoning_effort=medium (特定)")

ProviderProfile 已注册。
gpt-5.4 将使用: temperature=0 (继承) + reasoning_effort=medium (特定)


### 2.1 ProviderProfile 字段

| 字段 | 类型 | 说明 |
|------|------|------|
| `init_kwargs` | Mapping[str, Any] | 静态初始化参数，转发给 `init_chat_model` |
| `pre_init` | Callable[[str], None] | 构造前的副作用（如凭据验证） |
| `init_kwargs_factory` | Callable[[], dict[str, Any]] | 运行时派生的 kwargs（如从环境变量获取 headers） |

## 3. 注册键与查找顺序

两种 Profile 使用相同的键格式：

| 级别 | 示例 | 作用范围 |
|------|------|----------|
| Provider 级 | `"openai"` | 该 Provider 所有模型 |
| 模型级 | `"openai:gpt-5.4"` | 仅该特定模型 |

**查找顺序（预配置模型实例）：**
1. 精确 `provider:identifier` 匹配
2. Identifier-only（仅当 identifier 已含 `:`）
3. Provider-only 回退

**合并规则：** 两个级别都存在时，模型级覆盖 Provider 级（未设置的字段继承）。重新注册同一键会合并而非替换。

## 4. 合并语义（Merge Semantics）

| 字段 | 合并行为 |
|------|----------|
| `base_system_prompt`, `system_prompt_suffix` | 新值胜出，否则继承 |
| `tool_description_overrides` | 逐键合并，共享键上新值胜出 |
| `excluded_tools`, `excluded_middleware` | 集合并集 |
| `extra_middleware` | 按具体类合并：新实例替换同位置旧实例，新类追加 |
| `general_purpose_subagent` | 逐字段合并（未设置字段继承） |
| `init_kwargs` (Provider) | 字典逐键合并，共享键上新值胜出 |
| `pre_init` (Provider) | Callable 链式执行：现有关联先运行，再运行新的 |
| `init_kwargs_factory` (Provider) | 工厂链式执行，每次 `resolve_model` 调用时合并输出 |

## 5. 从配置文件加载

使用 `HarnessProfileConfig` 从 YAML/JSON 加载配置。

In [ ]:
import yaml
from deepagents import HarnessProfileConfig, register_harness_profile

# YAML 配置文件示例
yaml_content = """
base_system_prompt: You are helpful.
system_prompt_suffix: Respond briefly.
excluded_tools:
  - execute
  - grep
excluded_middleware:
  - SummarizationMiddleware
  - my_pkg.middleware:TelemetryMiddleware
general_purpose_subagent:
  enabled: false
"""

# 从 YAML 字符串加载
config = HarnessProfileConfig.from_dict(yaml.safe_load(yaml_content))

register_harness_profile("openai", config)

print("从 YAML 配置加载并注册 HarnessProfile 完成。")
print(f"base_system_prompt: {config.base_system_prompt}")
print(f"excluded_tools: {config.excluded_tools}")

从 YAML 配置加载并注册 HarnessProfile 完成。
base_system_prompt: You are helpful.
excluded_tools: {'execute', 'grep'}


### 反向导出

使用 `HarnessProfileConfig.from_harness_profile(...)` 将运行时 Profile 导出为可序列化的配置形状：

```python
# 从 HarnessProfile 导出
exported = HarnessProfileConfig.from_harness_profile(profile)
print(yaml.dump(exported.to_dict()))
```

注意：`extra_middleware` 非空或中间件类定义在 `__main__` / 函数作用域中时无法序列化。

## 6. 作为 Plugin 分发 Profile

通过 `importlib.metadata` entry points 自动注册 Profile，无需用户手动调用 `register_*_profile`。

**加载顺序：** 内置 Profile → Entry-point 插件 → 用户代码中的 `register_*_profile` 调用

**`pyproject.toml` 配置：**

```toml
[project.entry-points."deepagents.harness_profiles"]
my_provider = "my_pkg.profiles:register_harness"

[project.entry-points."deepagents.provider_profiles"]
my_provider = "my_pkg.profiles:register_provider"
```

**回调函数示例：**

In [ ]:
from deepagents import (
    HarnessProfile,
    ProviderProfile,
    register_harness_profile,
    register_provider_profile,
)

def register_harness() -> None:
    """为插件注册 HarnessProfile 的回调"""
    register_harness_profile(
        "my_provider",
        HarnessProfile(
            system_prompt_suffix="Batch independent tool calls in parallel.",
        ),
    )

def register_provider() -> None:
    """为插件注册 ProviderProfile 的回调"""
    register_provider_profile(
        "my_provider",
        ProviderProfile(init_kwargs={"temperature": 0}),
    )

print("Plugin 入口函数定义完成。")
print("说明：这些函数作为 entry point 回调，在 deepagents.profiles 导入时自动执行。")

Plugin 入口函数定义完成。
说明：这些函数作为 entry point 回调，在 deepagents.profiles 导入时自动执行。


## 7. 最佳实践

| 场景 | 建议 |
|------|------|
| 模型相关的 Prompt 调整 | 使用 `HarnessProfile` 的 `system_prompt_suffix` |
| 隐藏不兼容的工具 | 使用 `excluded_tools`（不要用 excluded_middleware 移除脚手架） |
| 全局不依赖模型的设置 | 直接在 `create_deep_agent` 调用处设置 |
| 模型构造参数 | 使用 `ProviderProfile` 的 `init_kwargs` |
| 运行时派生配置 | 使用 `ProviderProfile` 的 `init_kwargs_factory` |
| 凭据验证 | 使用 `ProviderProfile` 的 `pre_init` |
| 配置文件管理 | 使用 `HarnessProfileConfig.from_dict` 加载 YAML/JSON |
| 分发 Profile | 使用 `importlib.metadata` entry points 作为插件 |

---

**参考链接**
- [Profiles 文档](https://docs.langchain.com/oss/python/deepagents/profiles)
- [Harness 概览](https://docs.langchain.com/oss/python/deepagents/harness)
- [模型配置](https://docs.langchain.com/oss/python/deepagents/models)
- [自定义指南](https://docs.langchain.com/oss/python/deepagents/customization)
- [MCP 集成](https://docs.langchain.com/mcp)